# Reddit Research: Taoism
## Text Preprocessing

This notebook prepares the Reddit Taoism dataset for further linguistic and thematic analysis. The main goal is to clean and normalize the text data to ensure consistency and remove noise such as punctuation, links, special characters, or stopwords. The process involves lowercasing, tokenization, lemmatization, and other standard natural language processing (NLP) techniques. This step is crucial for producing reliable results in topic modeling, sentiment classification, and discourse analysis in the subsequent notebooks.

In [38]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 2.8 MB/s eta 0:00:05
     ---- ----------------------------------- 1.3/12.8 MB 3.0 MB/s eta 0:00:04
     ----- ---------------------------------- 1.8/12.8 MB 3.0 MB/s eta 0:00:04
     -------- ------------------------------- 2.6/12.8 MB 3.1 MB/s eta 0:00:04
     --------- ------------------------------ 3.1/12.8 MB 3.1 MB/s eta 0:00:04
     ----------- ---------------------------- 3.7/12.8 MB 3.0 MB/s eta 0:00:04
     ------------- -------------------------- 4.5/12.8 MB 3.1 MB/s eta 0:00:03
     --------------- ------------------------ 5.0/12.8 MB 3.1 MB/s eta 0:00:03
     ------------------ --------------------- 5.8/12.8 MB 3.0 MB/s eta 0:00:03
     ------------------- -------------------- 6.3/12.8 MB 3.0 MB/s eta 0:00:03
     --------------------- ------------------ 6.8/12.8 MB 3.0 MB/s eta 0:00:02
     ----------------------- ---------------- 7.6/12.8 MB 3

In [7]:
import spacy
import pandas as pd

In [8]:
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

We are going to proceed to load the dataset and check that everything is correct.

In [11]:
# Load raw comments
comments_df = pd.read_csv("../data/taoism_comments.csv")

In [27]:
print(comments_df.shape)
print(comments_df.columns)

(2010, 8)
Index(['post_id', 'comment_id', 'parent_id', 'author', 'body', 'score',
       'created_utc', 'depth'],
      dtype='object')


Now we are going to create a function to make a preliminary cleaning of the text.

In [32]:
# Preprocessing Function
def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Basic whitespace and newline cleanup
    text = text.strip().replace('\n', ' ').replace('\r', ' ')
    
    doc = nlp(text.lower())  # lowercase and tokenize
    tokens = [
        token.lemma_ for token in doc
        if not token.is_stop            # remove stopwords
        and not token.is_punct          # remove punctuation
        and not token.like_url          # remove links
        and not token.like_email        # remove emails
        and not token.is_space
        and len(token) > 2              # remove very short tokens
    ]
    return " ".join(tokens)

In [51]:
# Clean all comments
comments_df['clean_text'] = comments_df['body'].fillna("").apply(clean_text)

In [52]:
comments_df[['body', 'clean_text']].sample(5)

,body,clean_text
714,"""You think that's air you're breathing?""",think air breathe
283,"That's more about Selderij's ""flowery"" transla...",selderij flowery translation
1136,"I would say the answer is a resounding ""**YES""...",answer resounding yes different interpreter cr...
1782,"Yep, I did try to warn you of the futility of ...",yep try warn futility try educate tree hugger ...
757,That's perfect. I actually have a specific bea...,perfect actually specific beach use hear sound...


In [57]:
comments_df["original_length"] = comments_df["body"].str.split().str.len()
comments_df["clean_length"] = comments_df["clean_text"].str.split().str.len()

print("Avg. original length:", comments_df["original_length"].mean())
print("Avg. cleaned length:", comments_df["clean_length"].mean())

Avg. original length: 69.58457711442786
Avg. cleaned length: 29.103482587064676


In [59]:
# Word count distribution
comments_df["word_count"] = comments_df["clean_text"].apply(lambda x: len(x.split()))
comments_df["word_count"].describe()

count    2010.000000
mean       29.103483
std        47.987163
min         0.000000
25%         5.000000
50%        14.000000
75%        33.000000
max       551.000000
Name: word_count, dtype: float64

As we can appreciate, the function has reduced the average length from 69.58 to 29.10, which is less than half the original length. 

In [62]:
# For BERTopic input
texts_cleaned = comments_df['clean_text'].tolist()

# Saved cleaned dataframe
comments_df.to_csv("../data/taoism_comments_cleaned.csv", index=False)

In this notebook, we prepared the Reddit Taoism comments for computational analysis by applying a thorough text preprocessing pipeline. Using spaCy, we lemmatized the text and removed stopwords, punctuation, URLs, and other non-informative elements. The resulting cleaned text exhibits reduced noise and improved consistency, enabling more accurate results in later analyses. We also quantified word counts to understand the structure of the data, and finally transformed the cleaned comments into a list format suitable for input into BERTopic. With this, the dataset is now ready for unsupervised topic modeling in the next phase.